# National Tool Graphs

A standalone sandbox for developing charts for the national tool. This notebook reads generated card CSVs directly and does **not** import any functions from `national_tool_metrics`.

The charts support dry proofing, relocation, and flood protection. Relocation and flood-protection results are filtered by a user-selected urbanisation threshold; flood protection also uses a selected design return period.

## 1. Configure and load the ADM0 adaptation metrics

Set the country and adaptation option below. Relocation and flood protection require an urbanisation threshold, supplied as a label such as `remote_area` or a DUC code such as `11`. Flood protection additionally requires a design standard of 10, 20, 50, 100, or 200 years.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# User parameters
ISO3 = "MOZ"
ADAPTATION_OPTION = "flood_protection"  # "dry_proofing", "relocation", or "flood_protection"
URBANISATION_THRESHOLD = "21"  # Relocation/flood protection: label or DUC code
DESIGN_FLOOD_PROTECTION_YEARS = 10  # Flood protection only: 10, 20, 50, 100, or 200

SUPPORTED_ADAPTATION_OPTIONS = {
    "dry_proofing",
    "relocation",
    "flood_protection",
}
SUPPORTED_DESIGN_FLOOD_PROTECTION_YEARS = {10, 20, 50, 100, 200}
if ADAPTATION_OPTION not in SUPPORTED_ADAPTATION_OPTIONS:
    raise ValueError(
        f"ADAPTATION_OPTION must be one of {sorted(SUPPORTED_ADAPTATION_OPTIONS)}."
    )

WORKING_DIRECTORY = Path.cwd().resolve()
RELATIVE_DATA_PATH = (
    Path("results")
    / ISO3
    / "adaptation_outcomes"
    / f"{ISO3}_adm0_adaptation_outcomes_{ADAPTATION_OPTION}_metrics.csv"
)

for candidate_root in (WORKING_DIRECTORY, *WORKING_DIRECTORY.parents):
    candidate_path = candidate_root / RELATIVE_DATA_PATH
    if candidate_path.exists():
        REPO_ROOT = candidate_root
        DATA_PATH = candidate_path
        break
else:
    raise FileNotFoundError(
        f"Could not find {RELATIVE_DATA_PATH} from {WORKING_DIRECTORY} or its parents."
    )

all_metrics = pd.read_csv(DATA_PATH)
required_columns = {
    "admin_level",
    "card",
    "outcome_type",
    "metric",
    "statistic",
    "sector",
    "wealth_group",
    "unit",
    "value",
}
if ADAPTATION_OPTION in {"relocation", "flood_protection"}:
    required_columns.update(
        {"urbanisation_threshold", "urbanisation_threshold_code"}
    )
if ADAPTATION_OPTION == "flood_protection":
    required_columns.add("design_return_period_years")

missing_columns = required_columns.difference(all_metrics.columns)
if missing_columns:
    raise ValueError(f"The selected CSV is missing columns: {sorted(missing_columns)}")

file_cards = set(all_metrics["card"].dropna().astype(str))
if file_cards != {ADAPTATION_OPTION}:
    raise ValueError(
        f"Expected card {ADAPTATION_OPTION!r}, but the file contains {sorted(file_cards)}."
    )

if ADAPTATION_OPTION in {"relocation", "flood_protection"}:
    threshold_codes = pd.to_numeric(
        all_metrics["urbanisation_threshold_code"], errors="raise"
    ).astype(int)
    threshold_text = str(URBANISATION_THRESHOLD).strip()

    if isinstance(URBANISATION_THRESHOLD, (int, np.integer)) or threshold_text.isdigit():
        threshold_mask = threshold_codes.eq(int(threshold_text))
    else:
        threshold_mask = (
            all_metrics["urbanisation_threshold"]
            .astype(str)
            .str.strip()
            .str.lower()
            .eq(threshold_text.lower())
        )

    scenario_metrics = all_metrics.loc[threshold_mask].copy()
    if scenario_metrics.empty:
        available_thresholds = (
            all_metrics[["urbanisation_threshold_code", "urbanisation_threshold"]]
            .drop_duplicates()
            .sort_values("urbanisation_threshold_code")
        )
        raise ValueError(
            "Unknown urbanisation threshold. Available values are:\n"
            + available_thresholds.to_string(index=False)
        )

    selected_codes = pd.to_numeric(
        scenario_metrics["urbanisation_threshold_code"], errors="raise"
    ).astype(int).unique()
    selected_names = (
        scenario_metrics["urbanisation_threshold"].dropna().astype(str).unique()
    )
    if len(selected_codes) != 1 or len(selected_names) != 1:
        raise ValueError("The selected urbanisation threshold is not unique.")

    SELECTED_THRESHOLD_CODE = int(selected_codes[0])
    SELECTED_THRESHOLD_NAME = selected_names[0]
    threshold_label = SELECTED_THRESHOLD_NAME.replace("_", " ").title()
else:
    scenario_metrics = all_metrics.copy()
    SELECTED_THRESHOLD_CODE = None
    SELECTED_THRESHOLD_NAME = None
    threshold_label = None

if ADAPTATION_OPTION == "flood_protection":
    design_years_text = str(DESIGN_FLOOD_PROTECTION_YEARS).strip()
    if not design_years_text.isdigit():
        raise ValueError("DESIGN_FLOOD_PROTECTION_YEARS must be a whole number.")
    SELECTED_DESIGN_RETURN_PERIOD_YEARS = int(design_years_text)
    if (
        SELECTED_DESIGN_RETURN_PERIOD_YEARS
        not in SUPPORTED_DESIGN_FLOOD_PROTECTION_YEARS
    ):
        raise ValueError(
            "DESIGN_FLOOD_PROTECTION_YEARS must be one of "
            f"{sorted(SUPPORTED_DESIGN_FLOOD_PROTECTION_YEARS)}."
        )

    design_return_periods = pd.to_numeric(
        scenario_metrics["design_return_period_years"], errors="raise"
    ).astype(int)
    metrics = scenario_metrics.loc[
        design_return_periods.eq(SELECTED_DESIGN_RETURN_PERIOD_YEARS)
    ].copy()
    if metrics.empty:
        available_design_years = sorted(design_return_periods.unique())
        raise ValueError(
            f"No rows found for the selected design standard. Available: "
            f"{available_design_years}"
        )
    SCENARIO_LABEL = (
        f"flood protection - {threshold_label} (DUC{SELECTED_THRESHOLD_CODE}), "
        f"{SELECTED_DESIGN_RETURN_PERIOD_YEARS}-year design"
    )
elif ADAPTATION_OPTION == "relocation":
    metrics = scenario_metrics
    SELECTED_DESIGN_RETURN_PERIOD_YEARS = None
    SCENARIO_LABEL = f"relocation - {threshold_label} (DUC{SELECTED_THRESHOLD_CODE})"
else:
    metrics = scenario_metrics
    SELECTED_DESIGN_RETURN_PERIOD_YEARS = None
    SCENARIO_LABEL = "dry proofing"

print(f"Reading: {DATA_PATH}")
print(f"Selected scenario: {SCENARIO_LABEL}")

## 2. Select and check the economic-benefit data

Only the three component capital-stock sectors are plotted. The `total` row is excluded because it is the sum of those sectors. For relocation, all rows have already been restricted to the selected urbanisation threshold.

In [ ]:
sector_order = ["residential", "non_residential", "infrastructure"]
statistic_order = ["baseline", "adapted", "avoided"]

mask = (
    metrics["admin_level"].eq("ADM0")
    & metrics["card"].eq(ADAPTATION_OPTION)
    & metrics["outcome_type"].eq("economic_benefit")
    & metrics["metric"].eq("average_annual_loss")
    & metrics["sector"].isin(sector_order)
    & metrics["statistic"].isin(statistic_order)
)
aal_rows = metrics.loc[mask, ["sector", "statistic", "unit", "value"]].copy()

if aal_rows.duplicated(["sector", "statistic"]).any():
    raise ValueError("Expected one ADM0 value per sector and statistic.")
if set(aal_rows["unit"].dropna()) != {"usd_per_year"}:
    raise ValueError("Expected all selected AAL values to use 'usd_per_year'.")

aal_rows["value"] = pd.to_numeric(aal_rows["value"], errors="raise")
plot_data = (
    aal_rows.pivot(index="sector", columns="statistic", values="value")
    .reindex(index=sector_order, columns=statistic_order)
)

if plot_data.isna().any().any():
    raise ValueError("Missing baseline, adapted, or avoided AAL for at least one sector.")
if (plot_data < 0).any().any():
    raise ValueError("AAL values must be non-negative for this reduction chart.")
if not np.allclose(
    plot_data["baseline"],
    plot_data["adapted"] + plot_data["avoided"],
    rtol=1e-9,
    atol=1e-3,
):
    raise ValueError("Baseline AAL does not equal adapted plus avoided AAL.")

summary = (plot_data / 1_000_000).rename(
    columns={
        "baseline": "Baseline AAL (USDm/year)",
        "adapted": "AAL after adaptation (USDm/year)",
        "avoided": "Adaptation benefit (USDm/year)",
    }
)
summary.index = summary.index.str.replace("_", " ").str.title()
summary.round(2)

## 3. Plot baseline AAL and the adaptation benefit

The full grey width is baseline AAL. Teal is drawn over the portion avoided through the selected adaptation option; the unhighlighted remainder is AAL after adaptation.

In [ ]:
chart_data = plot_data / 1_000_000
sector_labels = ["Residential", "Non-residential", "Infrastructure"]
y_positions = np.arange(len(chart_data))

fig, ax = plt.subplots(figsize=(10, 5.2))

# Draw the complete baseline first, then overlay the avoided-loss segment
# from the adapted AAL boundary to the original baseline value.
ax.barh(
    y_positions,
    chart_data["baseline"],
    height=0.58,
    color="#CBD5E1",
    edgecolor="#64748B",
    linewidth=0.8,
    label="AAL after adaptation",
)
ax.barh(
    y_positions,
    chart_data["avoided"],
    left=chart_data["adapted"],
    height=0.58,
    color="#0F9D8A",
    edgecolor="#087568",
    linewidth=0.8,
    hatch="///",
    label="Adaptation benefit (avoided AAL)",
)

x_max = chart_data["baseline"].max()
for y, (_, values) in zip(y_positions, chart_data.iterrows()):
    baseline = values["baseline"]
    adapted = values["adapted"]
    avoided = values["avoided"]

    # This boundary is the new AAL after adaptation.
    ax.vlines(adapted, y - 0.34, y + 0.34, color="#0F172A", linewidth=1.1)
    # ax.text(
    #     baseline + x_max * 0.015,
    #     y,
    #     f"USD {baseline:,.1f}m baseline",
    #     va="center",
    #     ha="left",
    #     fontsize=9,
    #     color="#334155",
    # )

    # if avoided > 0:
    #     ax.text(
    #         adapted + avoided / 2,
    #         y,
    #         f"−USD {avoided:,.1f}m",
    #         va="center",
    #         ha="center",
    #         fontsize=9,
    #         fontweight="bold",
    #         color="white",
    #     )
    # else:
    #     ax.text(
    #         baseline - x_max * 0.012,
    #         y,
    #         "USD 0 benefit",
    #         va="center",
    #         ha="right",
    #         fontsize=8.5,
    #         color="#475569",
    #     )

ax.set_yticks(y_positions, labels=sector_labels)
ax.invert_yaxis()
ax.set_xlim(0, x_max * 1.27)
ax.set_xlabel("Average annual loss (USD millions per year)")
ax.set_title(
    f"The economic benefits of adaptation ({ADAPTATION_OPTION})",
    loc="left",
    fontsize=15,
    fontweight="bold",
    pad=18,
)
# ax.text(
#     0,
#     1.02,
#     "Kenya · ADM0 · River flooding · JRC model",
#     transform=ax.transAxes,
#     ha="left",
#     va="bottom",
#     fontsize=10,
#     color="#64748B",
# )
ax.grid(axis="x", color="#E2E8F0", linewidth=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
plt.show()

## 4. Select and check exposure by wealth quintile

This chart uses the social-benefit rows for Q1 through Q5. The national total is excluded so each bar represents one wealth quintile. For relocation, the selected urbanisation threshold is retained throughout.

In [ ]:
quintile_order = ["q1", "q2", "q3", "q4", "q5"]

quintile_mask = (
    metrics["admin_level"].eq("ADM0")
    & metrics["card"].eq(ADAPTATION_OPTION)
    & metrics["outcome_type"].eq("social_benefit")
    & metrics["metric"].eq("average_annual_flood_exposure")
    & metrics["wealth_group"].isin(quintile_order)
    & metrics["statistic"].isin(statistic_order)
)
exposure_rows = metrics.loc[
    quintile_mask, ["wealth_group", "statistic", "unit", "value"]
].copy()

if exposure_rows.duplicated(["wealth_group", "statistic"]).any():
    raise ValueError("Expected one ADM0 exposure value per quintile and statistic.")
if set(exposure_rows["unit"].dropna()) != {"people_per_year"}:
    raise ValueError("Expected all selected exposure values to use 'people_per_year'.")

exposure_rows["value"] = pd.to_numeric(exposure_rows["value"], errors="raise")
quintile_data = (
    exposure_rows.pivot(index="wealth_group", columns="statistic", values="value")
    .reindex(index=quintile_order, columns=statistic_order)
)

if quintile_data.isna().any().any():
    raise ValueError("Missing baseline, adapted, or avoided exposure for a quintile.")
if (quintile_data < 0).any().any():
    raise ValueError("Exposure values must be non-negative for this reduction chart.")
if not np.allclose(
    quintile_data["baseline"],
    quintile_data["adapted"] + quintile_data["avoided"],
    rtol=1e-9,
    atol=1e-3,
):
    raise ValueError("Baseline exposure does not equal adapted plus avoided exposure.")

exposure_summary = quintile_data.rename(
    columns={
        "baseline": "Baseline exposure (people/year)",
        "adapted": "Exposure after adaptation (people/year)",
        "avoided": "Avoided exposure (people/year)",
    }
)
exposure_summary.index = ["Q1", "Q2", "Q3", "Q4", "Q5"]
exposure_summary.round(0)

## 5. Plot the exposure reduction by wealth quintile

The complete outlined width is baseline average annual flood exposure. Teal highlights the people per year removed from exposure by the selected adaptation option, while the remaining grey portion is exposure after adaptation.

In [ ]:
# Thousands of people per year keeps the axis and labels compact.
quintile_chart_data = quintile_data / 1_000
quintile_labels = ["Quintile 1", "Quintile 2", "Quintile 3", "Quintile 4", "Quintile 5"]
quintile_y = np.arange(len(quintile_chart_data))

fig, ax = plt.subplots(figsize=(10, 6.2))

ax.barh(
    quintile_y,
    quintile_chart_data["baseline"],
    height=0.58,
    color="#CBD5E1",
    edgecolor="#64748B",
    linewidth=0.8,
    label="Exposure after adaptation",
)
ax.barh(
    quintile_y,
    quintile_chart_data["avoided"],
    left=quintile_chart_data["adapted"],
    height=0.58,
    color="#0F9D8A",
    edgecolor="#087568",
    linewidth=0.8,
    hatch="///",
    label="Adaptation benefit (avoided exposure)",
)

quintile_x_max = quintile_chart_data["baseline"].max()
# for y, (_, values) in zip(quintile_y, quintile_chart_data.iterrows()):
#     baseline = values["baseline"]
#     adapted = values["adapted"]
#     avoided = values["avoided"]

#     ax.vlines(adapted, y - 0.34, y + 0.34, color="#0F172A", linewidth=1.1)
#     ax.text(
#         baseline + quintile_x_max * 0.015,
#         y,
#         f"{baseline:,.1f}k baseline",
#         va="center",
#         ha="left",
#         fontsize=9,
#         color="#334155",
#     )
#     ax.text(
#         adapted + avoided / 2,
#         y,
#         f"−{avoided:,.1f}k",
#         va="center",
#         ha="center",
#         fontsize=9,
#         fontweight="bold",
#         color="white",
#     )

ax.set_yticks(quintile_y, labels=quintile_labels)
ax.invert_yaxis()
ax.set_xlim(0, quintile_x_max * 1.27)
ax.set_xlabel("Average annual flood exposure (thousands of people per year)")
ax.set_title(
    f"The social benefits of adaptation ({ADAPTATION_OPTION})",
    loc="left",
    fontsize=15,
    fontweight="bold",
    pad=18,
)
# ax.text(
#     0,
#     1.02,
#     "Kenya · ADM0 · River flooding · JRC model",
#     transform=ax.transAxes,
#     ha="left",
#     va="bottom",
#     fontsize=10,
#     color="#64748B",
# )
ax.grid(axis="x", color="#E2E8F0", linewidth=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
plt.show()

## 6. Plot baseline and adapted concentration curves

This section reads the combined country concentration-curve results file. The adaptation option, urbanisation threshold, and flood-protection design standard determine which adapted column is plotted against the shared protected baseline. The dashed diagonal is the line of equality.

In [ ]:
CONCENTRATION_CURVE_PATH = (
    REPO_ROOT
    / "results"
    / ISO3
    / "concentration_curves"
    / f"{ISO3}_concentration_curves.csv"
)
if not CONCENTRATION_CURVE_PATH.exists():
    raise FileNotFoundError(
        f"Concentration-curve results file not found: {CONCENTRATION_CURVE_PATH}"
    )

population_column = "cumulative_population_share"
baseline_column = "flood_risk__jrc__baseline_protected"
if ADAPTATION_OPTION == "dry_proofing":
    adapted_column = "flood_risk__jrc__dry_proofing"
elif ADAPTATION_OPTION == "relocation":
    adapted_column = (
        f"flood_risk__jrc__relocation_duc{SELECTED_THRESHOLD_CODE}"
    )
else:
    adapted_column = (
        f"flood_risk__jrc__flood_protection_"
        f"rp{SELECTED_DESIGN_RETURN_PERIOD_YEARS}_duc{SELECTED_THRESHOLD_CODE}"
    )

curve_source = pd.read_csv(CONCENTRATION_CURVE_PATH)
required_curve_columns = [population_column, baseline_column, adapted_column]
missing_curve_columns = set(required_curve_columns).difference(curve_source.columns)
if missing_curve_columns:
    raise ValueError(
        "The combined concentration-curve results file is missing columns: "
        f"{sorted(missing_curve_columns)}"
    )

curves = (
    curve_source[required_curve_columns]
    .rename(
        columns={
            baseline_column: "baseline",
            adapted_column: "adapted",
        }
    )
    .apply(pd.to_numeric, errors="raise")
)

if curves.empty or curves.isna().any().any():
    raise ValueError("Concentration curves must contain complete numeric values.")
if not np.isfinite(curves.to_numpy()).all():
    raise ValueError("Concentration curves must contain only finite values.")
if not curves.apply(lambda series: series.between(0, 1).all()).all():
    raise ValueError("All cumulative shares must lie between 0 and 1.")
if not curves.apply(lambda series: series.is_monotonic_increasing).all():
    raise ValueError("All concentration-curve columns must be monotonic increasing.")
if not np.allclose(curves.iloc[0], 0) or not np.allclose(curves.iloc[-1], 1):
    raise ValueError("Every concentration curve must run from (0, 0) to (1, 1).")

population_share = curves[population_column]
baseline_curve = curves["baseline"]
adapted_curve = curves["adapted"]

print(f"Concentration curves: {CONCENTRATION_CURVE_PATH}")
print(f"Adapted column: {adapted_column}")

fig, ax = plt.subplots(figsize=(6, 8))
ax.plot(
    population_share,
    population_share,
    color="#94A3B8",
    linestyle="--",
    linewidth=1.5,
    label="Line of equality",
)
ax.plot(
    population_share,
    baseline_curve,
    color="#64748B",
    linewidth=2.5,
    label="Baseline",
)
ax.plot(
    population_share,
    adapted_curve,
    color="#0F9D8A",
    linewidth=2.5,
    label="Adapted",
)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("Cumulative population share (lowest to highest wealth)")
ax.set_ylabel("Cumulative share of average annual flood exposure")
ax.set_title(
    f"Flood Risk Inequality - Concentration Curve ({ADAPTATION_OPTION})",
    loc="left",
    fontsize=15,
    fontweight="bold",
    pad=18,
)
ax.grid(color="#E2E8F0", linewidth=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
plt.show()